In [1]:
from arc_vae import ArcEncoder, ArcDecoder
from dall_e import map_pixels, unmap_pixels, load_model
from dall_e import Encoder

import torch
import torch.nn.functional as F

import onnxruntime as ort
import numpy as np

In [4]:
device = torch.device("cuda")
base: Encoder = load_model("../../agentic/checkpoints/encoder.pkl", device)
# enc: ArcEncoder = ArcEncoder(64, "https://cdn.openai.com/dall-e/encoder.pkl", device)

In [5]:
enc: ArcEncoder = ArcEncoder(
    64,
    n_hid=base.n_hid,
    n_blk_per_group=base.n_blk_per_group,
    input_channels=base.input_channels,
    vocab_size=base.vocab_size,
    device=base.device,
    requires_grad=base.requires_grad,
    use_mixed_precision=base.use_mixed_precision,
)

In [7]:
for param in enc.parameters():
    param.requires_grad = False
enc.eval()

# dummy_img = torch.randn(1, 3, 480, 640, device=device)
# dummy_img = torch.randn(1, 3, 120, 160, device="cuda")
dummy_img = torch.randn(1, 3, 240, 320, device=device)
torch.onnx.export(
    enc.cuda(),
    dummy_img,
    "./arc_enc_64.onnx",
    export_params=True,
    input_names=["pixels"],
    # output_names=[""],
    # dynamic_axes={"pixels": {2: "height", 3: "width"}},
    # dynamic_axes={"pixels": {0: "batch"}},
    dynamic_axes=None
)
print("ONNX model saved: enc.onnx")

/home/nvidia/.local/lib/python3.10/site-packages/dall_e/encoder.py:89: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if x.shape[1] != self.input_channels:


ONNX model saved: enc.onnx


In [ ]:
# trtexec --onnx=arc_enc_64.onnx --saveEngine=arc_enc_64.trt --workspace=4096 --fp16